# Esqueleto andante — verificador de citações

Clona o repositório numa versão fixa e chama a CLI. Nenhuma lógica do pipeline fica nas células.

Antes de rodar: anexe o secret `GITHUB_TOKEN`, ligue a internet, fixe o Environment, e adicione a competição (ou o dataset privado) em **Add Input**. Detalhes em `docs/guia_kaggle.md`.

In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient

REPO = "Roberto207/agente_jusbrasil"
VERSAO = "main"      # branch de desenvolvimento; trocar por tag (ex.: sub-001) na primeira submissão
DESTINO = "/kaggle/temp/agente_jusbrasil"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
subprocess.run(
    ["git", "clone", "--quiet", f"https://{token}@github.com/{REPO}.git", DESTINO],
    check=True,
)
subprocess.run(["git", "-C", DESTINO, "remote", "set-url", "origin",
                "https://github.com/{REPO}.git".format(REPO=REPO)], check=True)
subprocess.run(["git", "-C", DESTINO, "checkout", "--quiet", VERSAO], check=True)
del token

In [ ]:
%cd /kaggle/temp/agente_jusbrasil
!pip install --quiet -r requirements.txt
!pip install --quiet --no-deps -e .

In [ ]:
!python -m verificador ambiente

In [ ]:
from pathlib import Path

DADOS = None
for pasta in Path("/kaggle/input").glob("*"):
    if (pasta / "desafio1_bracis.db").exists() or (pasta / "txt").is_dir():
        DADOS = pasta
        break
if DADOS is None:
    raise SystemExit("Anexe a competição (Add Input → Competitions) ou o dataset privado com os dados oficiais.")

SAIDA = "/kaggle/working/runs"
print("dados:", DADOS)

!python -m verificador indexar  --dados {DADOS}
!python -m verificador rodar    --entrada {DADOS}/txt --run esqueleto --saida {SAIDA}
!python -m verificador avaliar  --run esqueleto --saida {SAIDA} --dados {DADOS}

In [ ]:
from pathlib import Path

saida = Path("/kaggle/working/runs/esqueleto")
print("saídas em", saida.resolve())
for caminho in sorted(saida.rglob("*")):
    if caminho.is_file():
        print(" -", caminho.relative_to(saida))